# Using the HuggingFace API
## NER training, testing and finetuning

useful resources:
* https://github.com/huggingface/notebooks/blob/main/examples/token_classification.ipynb

How to use this notebook:
* change the variables under 1. Imports and general settings > #main parameters
* run all cells 

## 0. Check that all requirements are fulfilled

* Is a GPU available? Can we use it?
  * use `nvidia-smi [-L]` (`free -m`, `htop`) for inspection
  * short notice to others (gitter, Standup)
* Are all necessary libraries installed and functioning?
  * in virtual environment (`pyenv activate <virtualenv>`) run `pip install transformers`, `pip install torch` (for PyTorch, TensorFlow would be an alternative), `pip install datasets` and install/upgrade further dependencies in case needed

## 1. Imports and general settings

In [1]:
#imports
from transformers import AutoTokenizer, AutoModelForTokenClassification, AutoModelForSequenceClassification
from transformers import pipeline
import torch
import json
from datetime import datetime

/data-ssd/sophie.schneider/pyenv/versions/hf-transformers-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#set GPU else CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


### Choose a base model
* [RoBERTa](https://huggingface.co/docs/transformers/model_doc/roberta)
* [ELECTRA](https://huggingface.co/docs/transformers/model_doc/electra)
* [BSB-BERT Modell](https://huggingface.co/dbmdz/bert-base-german-cased) `#german`
* Flair: https://huggingface.co/flair/ner-german bzw. https://huggingface.co/flair/ner-german-large `#ner` `#german`
* [hmBERT](https://huggingface.co/dbmdz/bert-base-historic-multilingual-cased) `#hist`
* [distilbert](https://huggingface.co/distilbert/distilbert-base-uncased)

In [3]:
#main parameters
task = "ner" # one of "ner", "pos" or "chunk" from token classification
model_checkpoint = "prajjwal1/bert-tiny" #tiny bert, good for testing purposes
batch_size = 16
dataset_name = "conll2003"
eval_strategy = "epoch"
learning_rate = 2e-5
num_train_epochs = 3
weight_decay = 0.01

In [4]:
#set path to model

now = datetime.now().strftime('%Y-%m-%d_%H-%M')
model_name = model_checkpoint.split("/")[-1]

model_path = f"{model_name}-finetuned-{task}-{now}"

In [5]:
#write all parameters to dict to save later
parameters_dict = dict(((k, eval(k)) for k in ("task","model_checkpoint", "batch_size", "dataset_name", "eval_strategy", "learning_rate", "num_train_epochs", "weight_decay")))
parameters_dict

{'task': 'ner',
 'model_checkpoint': 'prajjwal1/bert-tiny',
 'batch_size': 16,
 'dataset_name': 'conll2003',
 'eval_strategy': 'epoch',
 'learning_rate': 2e-05,
 'num_train_epochs': 3,
 'weight_decay': 0.01}

## 2. Finetuning a Model
see also: [https://huggingface.co/docs/transformers/training](https://huggingface.co/docs/transformers/training) 

1. load and inspect a dataset from HF directly / load and inspect a dataset from another source
2. instantiate `Tokenizer` and tokenize dataset
3. load base model
4. set training hyperparameters
5. set evaluation metric
6. create Trainer object
7. fine-tune by calling `train()`

### 2.1a Load and inspect a dataset from HF directly

Datasets for Pretraining:
* OCR Volltext Basis aus der OCR Pipeline `#hist`
* [German DBMDZ BERT Corpus](https://huggingface.co/datasets/stefan-it/german-dbmdz-bert-corpus) `#german`
  
Datasets for NER Finetuning:
* sbb_ner_data (nach Konsistenzchecks) `#german` `#hist` `#ner`
* [HisGermaNER](https://huggingface.co/datasets/stefan-it/HisGermaNER) `#german` `#hist` `#ner`
* https://github.com/NEISSproject/NERDatasets `#german` `(#hist)` `#ner`
* [HIPE](https://github.com/hipe-eval/HIPE-2022-data) `#german` `#multilingual` `#hist` `#ner`
* https://github.com/UB-Mannheim/reichsanzeiger-nlp `#german` `#hist` `#ner`

In [6]:
# load a dataset
from datasets import load_dataset
dataset = load_dataset(dataset_name)

In [7]:
# inspect dataset
dataset["train"][100]

{'id': '100',
 'tokens': ['Rabinovich',
  'is',
  'winding',
  'up',
  'his',
  'term',
  'as',
  'ambassador',
  '.'],
 'pos_tags': [21, 42, 39, 33, 29, 21, 15, 21, 7],
 'chunk_tags': [11, 21, 22, 15, 11, 12, 13, 11, 0],
 'ner_tags': [1, 0, 0, 0, 0, 0, 0, 0, 0]}

In [8]:
label_list = dataset["train"].features[f"{task}_tags"].feature.names
label_list

['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']

### 2.2b Align labels with tokenized text (for NER/Sequence Labeling)

"we need to do some processing on our labels as the input ids returned by the tokenizer are longer than the lists of labels our dataset contain"

In [9]:
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

label_all_tokens = True

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)

    labels = []
    for i, label in enumerate(examples[f"{task}_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            # Special tokens have a word id that is None. We set the label to -100 so they are automatically
            # ignored in the loss function.
            if word_idx is None:
                label_ids.append(-100)
            # We set the label for the first token of each word.
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            # For the other tokens in a word, we set the label to either the current label or -100, depending on
            # the label_all_tokens flag.
            else:
                label_ids.append(label[word_idx] if label_all_tokens else -100)
            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [10]:
tokenize_and_align_labels(dataset['train'][:5]) 
#TODO: fix "Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation."

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


{'input_ids': [[101, 7327, 19164, 2446, 2655, 2000, 17757, 2329, 12559, 1012, 102], [101, 2848, 13934, 102], [101, 9371, 2727, 1011, 5511, 1011, 2570, 102], [101, 1996, 2647, 3222, 2056, 2006, 9432, 2009, 18335, 2007, 2446, 6040, 2000, 10390, 2000, 18454, 2078, 2329, 12559, 2127, 6529, 5646, 3251, 5506, 11190, 4295, 2064, 2022, 11860, 2000, 8351, 1012, 102], [101, 2762, 1005, 1055, 4387, 2000, 1996, 2647, 2586, 1005, 1055, 15651, 2837, 14121, 1062, 9328, 5804, 2056, 2006, 9317, 10390, 2323, 4965, 8351, 4168, 4017, 2013, 3032, 2060, 2084, 3725, 2127, 1996, 4045, 6040, 2001, 24509, 1012, 102]], 'token_type_ids': [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1], [1, 1, 1, 1, 1, 1

In [11]:
tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)

### 2.3 load base model

In [12]:
model = AutoModelForTokenClassification.from_pretrained(model_checkpoint, num_labels=len(label_list))

Some weights of BertForTokenClassification were not initialized from the model checkpoint at prajjwal1/bert-tiny and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


### 2.4 set training hyperparameters and evaluation metric

In [13]:
from transformers import TrainingArguments, Trainer
import numpy as np
import evaluate
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer)

In [14]:
args = TrainingArguments(
    model_path, #TODO: adapt to sth like models_finetuned/...?
    eval_strategy = eval_strategy,
    learning_rate=learning_rate,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=num_train_epochs,
    weight_decay=weight_decay,
)

In [15]:
# evaluation
metric = evaluate.load("seqeval") #load_metric has been removed, see https://discuss.huggingface.co/t/cant-import-load-metric-from-datasets/107524/2

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # Remove ignored index (special tokens)
    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

"""
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)
"""

'\ndef compute_metrics(eval_pred):\n    logits, labels = eval_pred\n    predictions = np.argmax(logits, axis=-1)\n    return metric.compute(predictions=predictions, references=labels)\n'

In [16]:
# training

#training_args = TrainingArguments(output_dir="test_trainer", eval_strategy="epoch")

trainer = Trainer(
    model,
    args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

### 2.5 Finetuning

In [17]:
trainer.train()

##TODO: add Parameter-Efficient Finetuning (`peft` library)

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.903700,0.497504,0.420599,0.381922,0.400328,0.877786
2,0.473100,0.414550,0.526747,0.481374,0.503040,0.898693
3,0.421400,0.391375,0.543809,0.507551,0.525055,0.902712


TrainOutput(global_step=2634, training_loss=0.5496902813400929, metrics={'train_runtime': 26.625, 'train_samples_per_second': 1582.083, 'train_steps_per_second': 98.929, 'total_flos': 4772669068566.0, 'train_loss': 0.5496902813400929, 'epoch': 3.0})

### 2.6 Evaluation

In [18]:
#overall eval metrics

overall_results = trainer.evaluate()

In [19]:
# eval metrics for each category

predictions, labels, _ = trainer.predict(tokenized_datasets["validation"])
predictions = np.argmax(predictions, axis=2)

# Remove ignored index (special tokens)
true_predictions = [
    [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
    for prediction, label in zip(predictions, labels)
]
true_labels = [
    [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
    for prediction, label in zip(predictions, labels)
]

results_per_tag = metric.compute(predictions=true_predictions, references=true_labels)
results_per_tag

{'LOC': {'precision': np.float64(0.6304801670146137),
  'recall': np.float64(0.692131398013751),
  'f1': np.float64(0.6598689002184996),
  'number': np.int64(2618)},
 'MISC': {'precision': np.float64(0.5696022727272727),
  'recall': np.float64(0.3257514216084484),
  'f1': np.float64(0.4144702842377261),
  'number': np.int64(1231)},
 'ORG': {'precision': np.float64(0.5124411566913248),
  'recall': np.float64(0.37062256809338523),
  'f1': np.float64(0.4301439458086368),
  'number': np.int64(2056)},
 'PER': {'precision': np.float64(0.47651006711409394),
  'recall': np.float64(0.5148319050758076),
  'f1': np.float64(0.49493029150823825),
  'number': np.int64(3034)},
 'overall_precision': np.float64(0.5438091813496344),
 'overall_recall': np.float64(0.5075511802215013),
 'overall_f1': np.float64(0.5250549704895267),
 'overall_accuracy': 0.9027118051694283}

In [20]:
#TODO: save parameter settings and evaluation results in one dict
metadata = {"parameters":parameters_dict, "overall_results":overall_results, "results_per_tag":results_per_tag}

with open(model_path + '/metadata.json', 'w') as fp:
    json.dump(metadata, fp, default=str)

In [21]:
#trainer.save_model("path/to/model")